# Smartphone Price Prediction
## End-to-End Machine Learning Project (Regression)

**Goal:** predict a smartphone's **Final Price** from its brand, model, RAM, storage,
color, and whether it ships free, using the same professional workflow used in the
Hotel Booking Cancellation and Adult Census Income projects: data understanding ->
cleaning -> EDA -> encoding -> scaling -> model comparison -> hyperparameter tuning ->
feature selection -> final evaluation -> saved deployment artifacts.


# **Part 1 - Imports**

In [ ]:
# ============================================================
# Part 1.1 : Imports
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor, ExtraTreesRegressor, BaggingRegressor,
    AdaBoostRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor,
    VotingRegressor, StackingRegressor
)
# NOTE: `from sklearn.experimental import enable_hist_gradient_boosting` was dropped
# from the template - it is no longer needed (HistGradientBoostingRegressor has been
# a stable, non-experimental estimator since scikit-learn 1.0) and importing it will
# raise a ModuleNotFoundError on current scikit-learn versions.

from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import pickle
import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", None)


# **Part 2 - Load & Understand the Data**

In [ ]:
# ============================================================
# Part 2.1 : Load Dataset
# ============================================================

df = pd.read_csv("smartphones.csv")

df.head()


In [ ]:
# Part 2.2 : Shape
df.shape


In [ ]:
# Part 2.3 : Columns
df.columns


In [ ]:
# Part 2.4 : Data Types & Non-Null Counts
df.info()


In [ ]:
# Part 2.5 : Statistical Summary
df.describe(include="all")


In [ ]:
# Part 2.6 : Missing Values Per Column
df.isnull().sum().sort_values(ascending=False)


**Observation:** only `RAM` and `Storage` have missing values (~27% and ~1.4% of rows
respectively). Every other column, including the target `Final Price`, is fully populated.

In [ ]:
# Part 2.7 : Duplicate Rows
df.duplicated().sum()


No duplicate rows to remove.

# **Part 3 - Train / Test Split**

Splitting *before* imputation, encoding, and scaling avoids data leakage - every
preprocessing object below is fit only on the training set, then applied to the test
set, exactly like the other two projects.

In [ ]:
# ============================================================
# Part 3.1 : Split Features & Target
# ============================================================

X = df.drop("Final Price", axis=1)
y = df["Final Price"]


In [ ]:
# ============================================================
# Part 3.2 : Train / Test Split
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, shuffle=True, random_state=42
)

X_train.reset_index(drop=True, inplace=True)
X_test.reset_index(drop=True, inplace=True)
y_train.reset_index(drop=True, inplace=True)
y_test.reset_index(drop=True, inplace=True)

print("X_train:", X_train.shape, " X_test:", X_test.shape)


# **Part 4 - Handle Missing Values**

In [ ]:
# Part 4.1 : Missing Values in Train
X_train.isnull().sum().sort_values(ascending=False)


In [ ]:
# Part 4.2 : Check RAM Skewness (choose an imputation strategy)
plt.hist(X_train["RAM"])
plt.title("RAM Distribution")
plt.show()

print("RAM skew:", X_train["RAM"].skew())


**Decision:** `RAM` is right-skewed, so median imputation (robust to skew/outliers) is used rather than the mean.

In [ ]:
# ============================================================
# Part 4.3 : Impute RAM (Median) - fit on train, transform both
# ============================================================

ram_imputer = SimpleImputer(missing_values=np.nan, strategy="median")

X_train["RAM"] = ram_imputer.fit_transform(X_train[["RAM"]])
X_test["RAM"] = ram_imputer.transform(X_test[["RAM"]])


In [ ]:
# Part 4.4 : Check Storage Skewness
plt.hist(X_train["Storage"])
plt.title("Storage Distribution")
plt.show()

print("Storage skew:", X_train["Storage"].skew())


In [ ]:
# ============================================================
# Part 4.5 : Impute Storage (Median) - fit on train, transform both
# ============================================================

storage_imputer = SimpleImputer(missing_values=np.nan, strategy="median")

X_train["Storage"] = storage_imputer.fit_transform(X_train[["Storage"]])
X_test["Storage"] = storage_imputer.transform(X_test[["Storage"]])


In [ ]:
# Part 4.6 : Confirm No Missing Values Remain
X_train.isnull().sum().sum(), X_test.isnull().sum().sum()


# **Part 5 - Exploratory Data Analysis**

In [ ]:
# Part 5.1 : RAM vs Price
sns.scatterplot(x=X_train["RAM"], y=y_train)
plt.title("Final Price vs RAM")
plt.show()


In [ ]:
# Part 5.2 : RAM vs Price, colored by Storage
sns.scatterplot(x=X_train["RAM"], y=y_train, hue=X_train["Storage"])
plt.title("Final Price vs RAM (colored by Storage)")
plt.show()


In [ ]:
# Part 5.3 : RAM vs Price, colored by Brand
plt.figure(figsize=(10, 6))
sns.scatterplot(x=X_train["RAM"], y=y_train, hue=X_train["Brand"], legend=False)
plt.title("Final Price vs RAM (colored by Brand)")
plt.show()


In [ ]:
# Part 5.4 : Average Price by Brand & Storage
plt.figure(figsize=(20, 8))
sns.barplot(x=X_train["Brand"], y=y_train, hue=X_train["Storage"])
plt.xticks(rotation=90)
plt.title("Average Final Price by Brand & Storage")
plt.show()


In [ ]:
# Part 5.5 : Listings per Brand
plt.figure(figsize=(14, 5))
sns.countplot(x=X_train["Brand"])
plt.xticks(rotation=90)
plt.title("Number of Listings per Brand")
plt.show()


In [ ]:
# Part 5.6 : Listings per Storage Tier
sns.countplot(x=X_train["Storage"])
plt.xticks(rotation=90)
plt.title("Number of Listings per Storage Tier")
plt.show()


In [ ]:
# Part 5.7 : Listings per RAM Tier
sns.countplot(x=X_train["RAM"])
plt.xticks(rotation=90)
plt.title("Number of Listings per RAM Tier")
plt.show()


In [ ]:
# Part 5.8 : Target Distribution
plt.figure(figsize=(8, 5))
sns.histplot(y_train, kde=True)
plt.title("Final Price Distribution (Train Set)")
plt.show()

print("Final Price skew:", y_train.skew())


**EDA Insights:**
- Price rises clearly with **RAM** and **Storage** - the two strongest visible drivers of price.
- Brand has a big effect on price at the *same* RAM/Storage tier - some brands command a
  premium, others are consistently budget-oriented.
- Listings are heavily concentrated in a handful of RAM tiers (4/6/8 GB) and Storage
  tiers (64/128/256 GB), with a long tail of less common configurations.
- **`Final Price` itself is right-skewed** (a small number of expensive flagship phones
  pull the distribution's tail out) - worth keeping in mind when reading RMSE, since a
  handful of expensive phones can dominate the squared-error metrics.
- `Model` has very high cardinality (383 unique values across only 1,816 rows - about
  4-5 listings per model on average), which is worth remembering when the encoding
  choice for it is discussed next.

# **Part 6 - Categorical Cardinality**

In [ ]:
X_train.select_dtypes("object").nunique()


# **Part 7 - Smart Encoding Strategy**

Same feature-specific approach as the other two projects:

- **Drop `Smartphone`** - free-text listing title, not a reusable feature.
- **Label Encoding** - for `Brand` and `Model` (each gets its own encoder). Any category
  seen in the test set but not during training is mapped to a single dedicated
  **"unknown" bucket** instead of being assigned a new arbitrary ID per unseen value
  (a bug in the original notebook: it previously gave *different* unseen brands/models
  *different* IDs, which is misleading — the model never saw any of them during training,
  so they should all collapse into one consistent "unknown" category rather than implying
  a numeric relationship between them that doesn't exist).
- **Binary Encoding** - `Free` (`Yes`/`No`) via `LabelEncoder`.
- **One-Hot Encoding** - `Color` (only 17 categories, low cardinality, no natural order),
  fit on train and applied identically to test via `handle_unknown="ignore"`.

In [ ]:
# ============================================================
# Part 7.1 : Drop Non-Predictive Column
# ============================================================

X_train.drop("Smartphone", axis=1, inplace=True)
X_test.drop("Smartphone", axis=1, inplace=True)


In [ ]:
# ============================================================
# Part 7.2 : Label Encode Brand (with a single "unknown" bucket)
# ============================================================

brand_encoder = LabelEncoder()
X_train["Brand"] = brand_encoder.fit_transform(X_train["Brand"])

brand_map = {value: idx for idx, value in enumerate(brand_encoder.classes_)}
UNKNOWN_BRAND_ID = len(brand_encoder.classes_)  # one single fallback id for every unseen brand

X_test["Brand"] = X_test["Brand"].apply(lambda x: brand_map.get(x, UNKNOWN_BRAND_ID))


In [ ]:
# ============================================================
# Part 7.3 : Label Encode Free (Yes/No)
# ============================================================

free_encoder = LabelEncoder()
X_train["Free"] = free_encoder.fit_transform(X_train["Free"])
X_test["Free"] = free_encoder.transform(X_test["Free"])


In [ ]:
# ============================================================
# Part 7.4 : One-Hot Encode Color
# ============================================================

color_encoder = OneHotEncoder(handle_unknown="ignore")

train_colors = pd.DataFrame(
    color_encoder.fit_transform(X_train[["Color"]]).toarray(),
    columns=color_encoder.get_feature_names_out()
)
X_train = pd.concat([X_train.drop(columns=["Color"]), train_colors], axis=1)

test_colors = pd.DataFrame(
    color_encoder.transform(X_test[["Color"]]).toarray(),
    columns=color_encoder.get_feature_names_out()
)
X_test = pd.concat([X_test.drop(columns=["Color"]), test_colors], axis=1)

X_train.head()


In [ ]:
# ============================================================
# Part 7.5 : Label Encode Model (with a single "unknown" bucket)
# ============================================================
# NOTE: Model has 383 unique values across 1,816 rows (~5 rows/model), so a fair
# share of test-set models will not have appeared in training. Label encoding is
# kept here (matches the original notebook's approach and tree-based models can
# still exploit any partial signal in the arbitrary ordering), but be aware this is
# a weaker encoding choice for such high cardinality - see Part 13 for how much the
# final model actually ends up relying on it.

model_encoder = LabelEncoder()
X_train["Model"] = model_encoder.fit_transform(X_train["Model"])

model_map = {value: idx for idx, value in enumerate(model_encoder.classes_)}
UNKNOWN_MODEL_ID = len(model_encoder.classes_)

X_test["Model"] = X_test["Model"].apply(lambda x: model_map.get(x, UNKNOWN_MODEL_ID))


# **Part 8 - Correlation Heatmap**

In [ ]:
plt.figure(figsize=(20, 10))
sns.heatmap(
    pd.concat([X_train, y_train], axis=1).corr(),
    annot=True, cmap="coolwarm", fmt=".2f"
)
plt.title("Correlation Heatmap (Features & Final Price)")
plt.show()


# **Part 9 - Feature Scaling**

Feature names are captured *before* scaling and reattached afterward — `StandardScaler`
returns a plain NumPy array, and losing the column names here would make feature
importance and the deployment pipeline harder to interpret later (this was silently lost
in the original notebook).

In [ ]:
# ============================================================
# Part 9.1 : Scale Numerical Features (Keep Column Names)
# ============================================================

feature_names = X_train.columns.tolist()

scaler = StandardScaler()

X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=feature_names)
X_test = pd.DataFrame(scaler.transform(X_test), columns=feature_names)

X_train.head()


# **Part 10 - Regression Models Comparison**

Fourteen algorithms are trained and compared on identical data.

In [ ]:
# ============================================================
# Part 10.1 : Regression Models
# ============================================================

models = {
    "Linear Regression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "KNN": KNeighborsRegressor(n_neighbors=5, metric="minkowski", p=1),
    "SVM": SVR(kernel="linear", C=100, gamma=1),
    "Decision Tree": DecisionTreeRegressor(criterion="squared_error", max_depth=10, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, criterion="squared_error", max_depth=10, random_state=42),
    "Extra Trees": ExtraTreesRegressor(n_estimators=100, random_state=42),
    "Bagging": BaggingRegressor(
        estimator=DecisionTreeRegressor(criterion="squared_error", max_depth=10),
        n_estimators=50, random_state=42
    ),
    "AdaBoost": AdaBoostRegressor(n_estimators=50, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42),
    "Hist Gradient Boosting": HistGradientBoostingRegressor(max_iter=100, learning_rate=0.1, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, verbosity=0),
    "CatBoost": CatBoostRegressor(verbose=0, iterations=100, learning_rate=0.1, random_state=42),
    "LightGBM": LGBMRegressor(n_estimators=100, learning_rate=0.1, random_state=42, verbose=-1)
}

print(f"Total Models : {len(models)}")


In [ ]:
# ============================================================
# Part 10.2 : Metrics Function
# ============================================================

def regression_metrics(model, X, y_actual):
    y_pred = model.predict(X)
    MAE = mean_absolute_error(y_actual, y_pred)
    MSE = mean_squared_error(y_actual, y_pred)
    RMSE = np.sqrt(MSE)
    R2 = r2_score(y_actual, y_pred)
    return MAE, MSE, RMSE, R2


In [ ]:
# ============================================================
# Part 10.3 : Train & Evaluate Every Model
# ============================================================

results = {}
trained_models = {}

for model_name, model in tqdm(models.items()):

    model.fit(X_train, y_train)

    MAE_Train, MSE_Train, RMSE_Train, R2_Train = regression_metrics(model, X_train, y_train)
    MAE_Test, MSE_Test, RMSE_Test, R2_Test = regression_metrics(model, X_test, y_test)

    results[model_name] = {
        "MAE Train": MAE_Train, "MAE Test": MAE_Test,
        "MSE Train": MSE_Train, "MSE Test": MSE_Test,
        "RMSE Train": RMSE_Train, "RMSE Test": RMSE_Test,
        "R2 Train": R2_Train, "R2 Test": R2_Test
    }

    trained_models[model_name] = model


In [ ]:
# ============================================================
# Part 10.4 : Comparison Table
# ============================================================
# Sorted by R2 Test - unlike the classification projects, there is no class-
# imbalance distortion here, so R2 on the held-out test set is a fair, direct
# measure of how well each model explains price variance.

results_df = pd.DataFrame(results).T.sort_values(by="R2 Test", ascending=False)

results_df


In [ ]:
# Part 10.5 : Visual Comparison
results_df[["R2 Train", "R2 Test"]].plot.bar(figsize=(13, 6))
plt.title("Model Comparison - R2 Train vs R2 Test")
plt.axhline(0, color="black", linewidth=0.8)
plt.xticks(rotation=45, ha="right")
plt.show()


**Watch for overfitting:** a model with a much higher R² on train than on test
(e.g. an unpruned Decision Tree or an overly deep Random Forest) is memorizing training
rows rather than generalizing — the gap itself is a warning sign worth checking here,
the same way it flagged Random Forest / Extra Trees overfitting in the hotel project.

# **Part 11 - Voting & Stacking Ensembles**

In [ ]:
# ============================================================
# Part 11.1 : Voting Regressors
# ============================================================

clf1 = LinearRegression()
clf2 = RandomForestRegressor(n_estimators=100, random_state=42)
clf3 = DecisionTreeRegressor(max_depth=10, random_state=42)

voting_simple = VotingRegressor(estimators=[("lr", clf1), ("rf", clf2), ("dt", clf3)])
voting_weighted = VotingRegressor(estimators=[("lr", clf1), ("rf", clf2), ("dt", clf3)], weights=[1, 2, 1])

for name, vmodel in [("Voting (Simple Average)", voting_simple), ("Voting (Weighted)", voting_weighted)]:

    vmodel.fit(X_train, y_train)

    MAE_Train, MSE_Train, RMSE_Train, R2_Train = regression_metrics(vmodel, X_train, y_train)
    MAE_Test, MSE_Test, RMSE_Test, R2_Test = regression_metrics(vmodel, X_test, y_test)

    results[name] = {
        "MAE Train": MAE_Train, "MAE Test": MAE_Test,
        "MSE Train": MSE_Train, "MSE Test": MSE_Test,
        "RMSE Train": RMSE_Train, "RMSE Test": RMSE_Test,
        "R2 Train": R2_Train, "R2 Test": R2_Test
    }

    trained_models[name] = vmodel

print("Voting regressors trained.")


In [ ]:
# ============================================================
# Part 11.2 : Stacking Regressor
# ============================================================

base_regressors = [
    ("rf", RandomForestRegressor(n_estimators=100, random_state=42)),
    ("gb", GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)),
    ("cat", CatBoostRegressor(iterations=100, learning_rate=0.1, verbose=0, random_state=42)),
    ("lgb", LGBMRegressor(n_estimators=100, learning_rate=0.1, random_state=42, verbose=-1))
]

meta_regressor = XGBRegressor(random_state=42, verbosity=0)

stacking_regressor = StackingRegressor(
    estimators=base_regressors,
    final_estimator=meta_regressor,
    cv=5,
    passthrough=True,
    n_jobs=-1
)

stacking_regressor.fit(X_train, y_train)

MAE_Train, MSE_Train, RMSE_Train, R2_Train = regression_metrics(stacking_regressor, X_train, y_train)
MAE_Test, MSE_Test, RMSE_Test, R2_Test = regression_metrics(stacking_regressor, X_test, y_test)

results["Stacking"] = {
    "MAE Train": MAE_Train, "MAE Test": MAE_Test,
    "MSE Train": MSE_Train, "MSE Test": MSE_Test,
    "RMSE Train": RMSE_Train, "RMSE Test": RMSE_Test,
    "R2 Train": R2_Train, "R2 Test": R2_Test
}

trained_models["Stacking"] = stacking_regressor

print("Stacking regressor trained.")


In [ ]:
# ============================================================
# Part 11.3 : Full Comparison Table (Base Models + Ensembles)
# ============================================================

results_df = pd.DataFrame(results).T.sort_values(by="R2 Test", ascending=False)

results_df


In [ ]:
# Part 11.4 : Full Metric Comparison Plots
cols = [
    ["MAE Train", "MAE Test"],
    ["MSE Train", "MSE Test"],
    ["RMSE Train", "RMSE Test"],
    ["R2 Train", "R2 Test"]
]

plt.figure(figsize=(20, 20))
for i, col in enumerate(cols):
    ax = plt.subplot(2, 2, i + 1)
    results_df[col].plot.bar(ax=ax)
    ax.set_title(col[0].split(" ")[0] + " Comparison")
    ax.tick_params(axis="x", labelrotation=90)
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# Part 11.5 : Best Model
# ============================================================

best_model_name = results_df.index[0]
best_model = trained_models[best_model_name]

print("="*60)
print("Best Model :", best_model_name)
print("R2 Test    :", results_df.loc[best_model_name, "R2 Test"])
print("R2 Train   :", results_df.loc[best_model_name, "R2 Train"])
print("="*60)


# **Part 12 - Hyperparameter Tuning (RandomizedSearchCV)**

`RandomizedSearchCV` is used instead of an exhaustive grid search - faster, and it
explores a wider parameter space. If a Voting or Stacking ensemble happens to win Part
11.5, tuning is skipped (they are already combinations of the tuned/default base models
above), and the individual base learners' comparison results are used as-is.

In [ ]:
# ============================================================
# Part 12.1 : Parameter Grid Per Model Type
# ============================================================

TUNABLE = best_model_name not in ["Voting (Simple Average)", "Voting (Weighted)", "Stacking"]

if not TUNABLE:
    print(f"'{best_model_name}' is an ensemble of the models above - skipping further tuning.")
    param_dist = {}

elif best_model_name == "CatBoost":
    param_dist = {
        "depth": [4, 6, 8, 10],
        "learning_rate": [0.01, 0.03, 0.05, 0.1],
        "iterations": [200, 400, 600],
        "l2_leaf_reg": [1, 3, 5, 7]
    }

elif best_model_name == "LightGBM":
    param_dist = {
        "n_estimators": [100, 200, 400],
        "learning_rate": [0.01, 0.03, 0.05, 0.1],
        "max_depth": [-1, 5, 10, 15],
        "num_leaves": [15, 31, 63]
    }

elif best_model_name == "XGBoost":
    param_dist = {
        "n_estimators": [100, 200, 400],
        "learning_rate": [0.01, 0.03, 0.05, 0.1],
        "max_depth": [3, 5, 7, 9],
        "subsample": [0.7, 0.85, 1.0]
    }

elif best_model_name in ["Random Forest", "Extra Trees"]:
    param_dist = {
        "n_estimators": [100, 200, 400],
        "max_depth": [None, 10, 20, 30],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2", None]
    }

elif best_model_name == "Gradient Boosting":
    param_dist = {
        "n_estimators": [100, 200, 300],
        "learning_rate": [0.01, 0.05, 0.1],
        "max_depth": [2, 3, 4, 5]
    }

elif best_model_name == "Hist Gradient Boosting":
    param_dist = {
        "max_iter": [100, 200, 300],
        "learning_rate": [0.01, 0.05, 0.1],
        "max_depth": [None, 5, 10, 20]
    }

elif best_model_name == "Decision Tree":
    param_dist = {
        "max_depth": [None, 5, 10, 20, 30],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4]
    }

elif best_model_name == "Bagging":
    param_dist = {
        "n_estimators": [30, 50, 100],
        "max_samples": [0.6, 0.8, 1.0],
        "max_features": [0.6, 0.8, 1.0]
    }

elif best_model_name == "AdaBoost":
    param_dist = {
        "n_estimators": [30, 50, 100, 150],
        "learning_rate": [0.01, 0.05, 0.1, 0.5, 1.0]
    }

elif best_model_name in ["Lasso", "Ridge"]:
    param_dist = {"alpha": [0.001, 0.01, 0.1, 1, 10, 100]}

elif best_model_name == "KNN":
    param_dist = {"n_neighbors": [3, 5, 7, 9, 11, 15], "p": [1, 2]}

else:
    param_dist = {}
    print(f"No tuning grid defined for {best_model_name} - using default params.")


In [ ]:
# ============================================================
# Part 12.2 : Run RandomizedSearchCV
# ============================================================

if TUNABLE and param_dist:

    random_search = RandomizedSearchCV(
        estimator=best_model,
        param_distributions=param_dist,
        n_iter=20,
        scoring="r2",
        cv=5,
        random_state=42,
        n_jobs=-1,
        verbose=1
    )

    random_search.fit(X_train, y_train)

    print("Best Params :", random_search.best_params_)
    print("Best CV R2  :", random_search.best_score_)

else:
    class _NoTuning:
        best_params_ = {}
    random_search = _NoTuning()


# **Part 13 - Feature Importance & Final Model**

In [ ]:
# ============================================================
# Part 13.1 : Build & Train the Final Model
# ============================================================

REGRESSOR_BY_NAME = {
    "Linear Regression": lambda p: LinearRegression(**p),
    "Lasso": lambda p: Lasso(**p, random_state=42),
    "Ridge": lambda p: Ridge(**p, random_state=42),
    "KNN": lambda p: KNeighborsRegressor(**p),
    "Decision Tree": lambda p: DecisionTreeRegressor(**p, random_state=42),
    "Random Forest": lambda p: RandomForestRegressor(**p, random_state=42),
    "Extra Trees": lambda p: ExtraTreesRegressor(**p, random_state=42),
    "Bagging": lambda p: BaggingRegressor(**p, random_state=42),
    "AdaBoost": lambda p: AdaBoostRegressor(**p, random_state=42),
    "Gradient Boosting": lambda p: GradientBoostingRegressor(**p, random_state=42),
    "Hist Gradient Boosting": lambda p: HistGradientBoostingRegressor(**p, random_state=42),
    "XGBoost": lambda p: XGBRegressor(**p, random_state=42, verbosity=0),
    "CatBoost": lambda p: CatBoostRegressor(**p, random_state=42, verbose=0),
    "LightGBM": lambda p: LGBMRegressor(**p, random_state=42, verbose=-1),
}

if not TUNABLE:
    # Best model was an ensemble (Voting / Stacking) - reuse it directly, it is
    # already fit on X_train/y_train from Part 11.
    final_model = best_model

elif best_model_name in REGRESSOR_BY_NAME:
    final_model = REGRESSOR_BY_NAME[best_model_name](random_search.best_params_)
    final_model.fit(X_train, y_train)

else:
    raise ValueError(f"Final model is not configured for '{best_model_name}'.")

print("Final Model :", best_model_name)


In [ ]:
# ============================================================
# Part 13.2 : Feature Importance
# ============================================================

if hasattr(final_model, "feature_importances_"):
    importance = pd.DataFrame({
        "Feature": X_train.columns,
        "Importance": final_model.feature_importances_
    }).sort_values(by="Importance", ascending=False).reset_index(drop=True)

    plt.figure(figsize=(9, 8))
    sns.barplot(data=importance, x="Importance", y="Feature", color="teal")
    plt.title(f"Feature Importance - {best_model_name}")
    plt.show()

    importance

else:
    importance = None
    print(f"'{best_model_name}' does not expose feature_importances_ "
          f"(e.g. it's a linear model or an ensemble) - skipping this plot.")


**Check here:** given `Model` has 383 unique values on only 1,816 rows (Part 6), see
how much importance the final model actually places on it versus `RAM`/`Storage`/`Brand` -
if it ranks unexpectedly high, that is more likely memorization of specific rows than a
generalizable pattern, and would be worth dropping or replacing with a smarter encoding
(e.g. target/frequency encoding) in a future iteration.

In [ ]:
# ============================================================
# Part 13.3 : Select Top Features
# ============================================================

if importance is not None:
    best_features = importance[importance["Importance"] > 0]["Feature"].tolist()
else:
    best_features = X_train.columns.tolist()

print(f"Selected {len(best_features)} / {X_train.shape[1]} features:")
print(best_features)

X_train_final = X_train[best_features]
X_test_final = X_test[best_features]


In [ ]:
# ============================================================
# Part 13.4 : Retrain Final Model on Selected Features
# ============================================================

if not TUNABLE:
    # Ensembles were trained on the full feature set in Part 11 - keep as-is
    # rather than refit (Voting/Stacking base learners expect consistent columns).
    pass
else:
    final_model = REGRESSOR_BY_NAME[best_model_name](random_search.best_params_)
    final_model.fit(X_train_final, y_train)


# **Part 14 - Final Model Evaluation**

In [ ]:
# ============================================================
# Part 14.1 : Final Metrics
# ============================================================

eval_X_test = X_test_final if TUNABLE else X_test

MAE_Train, MSE_Train, RMSE_Train, R2_Train = regression_metrics(
    final_model, X_train_final if TUNABLE else X_train, y_train
)
MAE_Test, MSE_Test, RMSE_Test, R2_Test = regression_metrics(final_model, eval_X_test, y_test)

final_metrics = {
    "MAE Train": MAE_Train, "MAE Test": MAE_Test,
    "MSE Train": MSE_Train, "MSE Test": MSE_Test,
    "RMSE Train": RMSE_Train, "RMSE Test": RMSE_Test,
    "R2 Train": R2_Train, "R2 Test": R2_Test
}

for k, v in final_metrics.items():
    print(f"{k:12s}: {v:,.3f}")


In [ ]:
# Part 14.2 : Actual vs Predicted (First 30 Test Samples)
y_pred = final_model.predict(eval_X_test)

plt.figure(figsize=(12, 6))
plt.plot(range(30), y_test.values[:30], label="Actual Price", marker="o")
plt.plot(range(30), y_pred[:30], label="Predicted Price", marker="x")
plt.xlabel("Sample Index")
plt.ylabel("Final Price")
plt.title(f"Actual vs Predicted Final Price - {best_model_name} (First 30 Test Samples)")
plt.legend()
plt.grid()
plt.show()


In [ ]:
# Part 14.3 : Predicted vs Actual Scatter (Full Test Set)
plt.figure(figsize=(7, 7))
plt.scatter(y_test, y_pred, alpha=0.4)
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    color="red", linestyle="--", label="Perfect Prediction"
)
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Predicted vs Actual Final Price (Full Test Set)")
plt.legend()
plt.show()


In [ ]:
# Part 14.4 : Residual Plot
residuals = y_test.values - y_pred

plt.figure(figsize=(8, 5))
plt.scatter(y_pred, residuals, alpha=0.4)
plt.axhline(0, color="red", linestyle="--")
plt.xlabel("Predicted Price")
plt.ylabel("Residual (Actual - Predicted)")
plt.title("Residual Plot")
plt.show()


**How to read this:** residuals scattered randomly around zero (no funnel shape,
no curve) indicate the model's errors are roughly consistent across price ranges. A
widening funnel toward higher predicted prices would mean the model is less reliable on
expensive flagship phones than on budget ones — worth checking given `Final Price` is
right-skewed (Part 5.8).

# **Part 15 - Save Deployment Artifacts**

Same artifact-saving pattern as the other two projects, ready for a Streamlit (or any
other) app to load and replay this exact preprocessing pipeline on a new phone listing.

In [ ]:
# ============================================================
# Part 15.1 : Save Model & Preprocessing Objects
# ============================================================

with open("final_model.pkl", "wb") as f:
    pickle.dump(final_model, f)

with open("ram_imputer.pkl", "wb") as f:
    pickle.dump(ram_imputer, f)

with open("storage_imputer.pkl", "wb") as f:
    pickle.dump(storage_imputer, f)

with open("brand_encoder.pkl", "wb") as f:
    pickle.dump({"encoder": brand_encoder, "map": brand_map, "unknown_id": UNKNOWN_BRAND_ID}, f)

with open("model_encoder.pkl", "wb") as f:
    pickle.dump({"encoder": model_encoder, "map": model_map, "unknown_id": UNKNOWN_MODEL_ID}, f)

with open("free_encoder.pkl", "wb") as f:
    pickle.dump(free_encoder, f)

with open("color_encoder.pkl", "wb") as f:
    pickle.dump(color_encoder, f)

with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

with open("best_features.pkl", "wb") as f:
    pickle.dump(best_features, f)

with open("feature_names.pkl", "wb") as f:
    pickle.dump(feature_names, f)

print("All artifacts saved successfully.")


# **Part 16 - Business Insights & Conclusion**

- **RAM and Storage are the two strongest, most direct price drivers** — both show a
  clear upward relationship with `Final Price` in the EDA scatterplots.
- **Brand carries real pricing power independent of specs** — at matching RAM/Storage
  tiers, average price still varies noticeably by brand (Part 5.4), meaning brand
  reputation/premium positioning is captured as a genuine signal, not just noise.
- **`Model` is a high-cardinality, low-signal risk** — 383 unique models across only 1,816
  rows means many test-set models were unseen during training (handled via a single
  "unknown" bucket, Part 7.5). Its actual importance was checked directly in Part 13.2
  rather than assumed.
- **`Final Price` is right-skewed** — a small number of expensive flagship phones pull the
  distribution's tail out, so RMSE (which penalizes large errors quadratically) is
  naturally higher than MAE; both were reported together instead of relying on RMSE alone.
- **Models were compared with both R² and the train/test gap in view**, not R² alone —
  unpruned trees and forests can look strong on Test R² while still overfitting, so the
  gap itself was flagged as a diagnostic (Part 10.5), the same lesson learned from the
  Random Forest overfitting seen in the hotel-booking project.

This notebook is now structured for direct reuse in a deployment app: `final_model.pkl`
plus every preprocessing object (`ram_imputer.pkl`, `storage_imputer.pkl`,
`brand_encoder.pkl`, `model_encoder.pkl`, `free_encoder.pkl`, `color_encoder.pkl`,
`scaler.pkl`, `best_features.pkl`, `feature_names.pkl`) are saved and ready to be loaded
the same way `app.py` does in the hotel-booking project.
